# 07b — Retrieval Test

In [1]:
!pip install -q faiss-cpu sentence-transformers huggingface_hub

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.5/18.5 MB 87.8 MB/s eta 0:00:00


In [2]:
import faiss
import json
import numpy as np
from sentence_transformers import SentenceTransformer
from huggingface_hub import hf_hub_download, login
from kaggle_secrets import UserSecretsClient
import torch

## 1. Konfigurasi

In [3]:
HF_REPO_ID  = "Makaareeem/publikasi-rag-knowledge-base"
MODEL_ID    = "nomic-ai/nomic-embed-text-v1.5"
DEVICE      = "cuda" if torch.cuda.is_available() else "cpu"
TOP_K       = 5
print(f"Device: {DEVICE}")

Device: cuda


## 2. Load Artefak dari HF

In [4]:
secrets = UserSecretsClient()
login(token=secrets.get_secret("HF_Faiss"))

index_path = hf_hub_download(repo_id=HF_REPO_ID, filename="faiss_index.bin", repo_type="dataset")
meta_path  = hf_hub_download(repo_id=HF_REPO_ID, filename="chunk_metadata.jsonl", repo_type="dataset")

index = faiss.read_index(index_path)
with open(meta_path) as f:
    meta = [json.loads(line) for line in f]

print(f"index.ntotal : {index.ntotal}")
print(f"metadata rows: {len(meta)}")

faiss_index.bin:   0%|          | 0.00/71.7M [00:00<?, ?B/s]

chunk_metadata.jsonl:   0%|          | 0.00/43.9M [00:00<?, ?B/s]

index.ntotal : 23336
metadata rows: 23336


## 3. Load Model

In [5]:
model = SentenceTransformer(MODEL_ID, trust_remote_code=True, device=DEVICE)

modules.json:   0%|          | 0.00/255 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/140 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/58.0 [00:00<?, ?B/s]

config.json: 0.00B [00:00, ?B/s]

...

<All keys matched successfully>


tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/695 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/286 [00:00<?, ?B/s]

## 4. Retrieval Test

In [6]:
def retrieve(query, k=TOP_K):
    q_vec = model.encode(
        [f"search_query: {query}"],
        normalize_embeddings=True,
        convert_to_numpy=True
    ).astype(np.float32)
    scores, ids = index.search(q_vec, k)
    results = []
    for score, fid in zip(scores[0], ids[0]):
        c = meta[fid]
        results.append({
            "rank":           len(results) + 1,
            "score":          round(float(score), 4),
            "chunk_id":       c["chunk_id"],
            "document_title": c["document_title"],
            "document_year":  c["document_year"],
            "page_start":     c["page_start"],
            "bps_url":        c["bps_url"],
            "text_preview":   c["text"][:200],
        })
    return results

In [7]:
queries = [
    "Berapa tingkat pengangguran terbuka di Indonesia?",
    "Bagaimana tren kemiskinan di Indonesia 2023?",
    "Apa indikator pembangunan manusia di Indonesia?",
]

for q in queries:
    print(f"\nQuery: {q}")
    print("─" * 70)
    for r in retrieve(q):
        print(f"  [{r['rank']}] score={r['score']}  {r['chunk_id']}")
        print(f"       {r['document_title']} ({r['document_year']}) | hal {r['page_start']}")
        print(f"       {r['text_preview']}...")
        print()


Query: Berapa tingkat pengangguran terbuka di Indonesia?
──────────────────────────────────────────────────────────────────────
  [1] score=0.7857  statistik-pemuda-indonesia-2025__sec_066__c001
       Statistik Pemuda Indonesia 2025 (2025) | hal 47
       Tingkat Pengangguran Terbuka (TPT)
adalah persentase angkatan kerja yang tidak mempunyai pekerjaan, sedang aktif mencari pekerjaan atau mempersiapkan usaha, mereka yang putus asa mencari pekerjaan dan...

  [2] score=0.7855  statistik-pemuda-indonesia-2024__sec_064__c001
       Statistik Pemuda Indonesia 2024 (2024) | hal 55
       Tingkat Pengangguran Terbuka (TPT)
adalah persentase angkatan kerja yang tidak mempunyai pekerjaan, sedang aktif mencari pekerjaan atau mempersiapkan usaha, mereka yang putus asa mencari
liii

pekerja...

  [3] score=0.7833  statistik-pemuda-indonesia-2023__5__c007
       Statistik Pemuda Indonesia 2023 (2023) | hal 263
       pengangguran terdidik. Tingginya tingkat pengangguran pada pemuda terdidik ters